# StateCraft Training Notebook (Colab)

Run this notebook in Google Colab to train StateCraft and export artifacts for sharing.

In [ ]:
# ===== 1) USER CONFIG =====
REPO_URL = "https://github.com/KanishkJaiswal-111/StateCraft.git"  # change if needed
BRANCH = "main"
WORKDIR = "/content/StateCraft"
RUN_ROOT = "/content/statecraft_runs"

# Training controls
    "TRAIN_MODE = \"grpo\"   # one of: grpo, standard, curriculum\n",
PPO_EPISODES = 2000
STANDARD_EPISODES = 500
CURRICULUM_EPISODES = 500
SCENARIO = "pandemic"

# Optional: mount Drive to persist outputs
USE_DRIVE = True
DRIVE_RUN_ROOT = "/content/drive/MyDrive/StateCraft_runs"

print("Config loaded. TRAIN_MODE=", TRAIN_MODE)

In [ ]:
# ===== 2) RUNTIME SETUP =====
import os, sys, json, shutil, subprocess, pathlib

def run(cmd):
    print("+", " \
,
,
,
,
,

os.chdir(WORKDIR)
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch check failed:", e)

run([sys.executable, "verify_integration.py"])
print("Setup complete.")

In [ ]:
# ===== 3) TRAIN =====
import time
from datetime import datetime

ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
RUN_DIR = os.path.join(RUN_ROOT, f"run_{ts}_{TRAIN_MODE}")
os.makedirs(RUN_DIR, exist_ok=True)
print("Run dir:", RUN_DIR)

if TRAIN_MODE == "grpo":
    from training.grpo_trainer import trainer, model, tokenizer
    
    ckpt_dir = os.path.join(RUN_DIR, "grpo_checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    
    print("Starting GRPO LLM training with Unsloth and TRL...")
    run([sys.executable, "-m", "training.grpo_trainer"])
    
    with open(os.path.join(RUN_DIR, "summary.json"), "w", encoding="utf-8") as f:
        json.dump({
            "mode": "grpo",
            "run_dir": RUN_DIR
        }, f, indent=2)

elif TRAIN_MODE == "standard":
    cmd = [
        sys.executable, "-m", "scripts.train_cloud",
        "--episodes", str(STANDARD_EPISODES),
        "--scenario", SCENARIO,
        "--output-dir", RUN_DIR,
    ]
    run(cmd)

elif TRAIN_MODE == "curriculum":
    cmd = [
        sys.executable, "-m", "scripts.train_cloud",
        "--curriculum",
        "--episodes", str(CURRICULUM_EPISODES),
        "--scenario", SCENARIO,
        "--output-dir", RUN_DIR,
    ]
    run(cmd)

else:
    "    raise ValueError(\"TRAIN_MODE must be one of: grpo, standard, curriculum\")\n",

print("Training complete.")


In [ ]:
    "# ===== 4) GENERALIZATION EVAL =====\n",
    "from eval.generalization import run_generalization_test\n",
    "\n",
    "gen = run_generalization_test()\n",
    "with open(os.path.join(RUN_DIR, \"generalization_results.json\"), \"w\", encoding=\"utf-8\") as f:\n",
    "    json.dump(gen, f, indent=2)\n",
    "print(\"Saved generalization_results.json\")\n"


In [ ]:
# ===== 5) PACKAGE OUTPUTS FOR SHARING =====
zip_base = os.path.join('/content', pathlib.Path(RUN_DIR).name)
zip_path = shutil.make_archive(zip_base, 'zip', RUN_DIR)
print('Packaged:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('Auto download unavailable in this runtime:', e)

print('Run directory:', RUN_DIR)